# Labeling sentiments

VADER was selected for sentiment labeling because it was specifically designed for informal texts (like reviews), where punctuation and capitalization have emotional meaning. "Amazing!!!" scores higher than "amazing" which is exactly what we need for this project.  

The compound score threshold was set at ±0.3 instead of the default ±0.05 to reduce noise in the neutral class. After testing, reviews with a compound score between -0.05 and 0.05 were producing inconsistent labels so rasing the treshold to ±0.3 produced cleaner and more reliable classification.

In [29]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from tqdm.auto import tqdm
tqdm.pandas()

In [2]:
pd.set_option('display.max_colwidth',None)
df = pd.read_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\airbnb_english_cleaned.csv")
df.head(2)

,date,reviewer_name,comments,language,exclamation_count,question_count,caps_ratio,word_count,cleaned_comments
0,3/30/2009,Lam,Daniel is really cool. The place was nice and clean. Very quiet neighborhood. He had maps and a lonely planet guide book in the room for you to use. I didnt have any trouble finding the place from Central Station. I would defintely come back! Thanks!,en,2,0,0.03600,46,daniel be really cool. the place be nice and clean. very quiet neighborhood. he have map and a lonely planet guide book in the room for you to use. i do not have any trouble find the place from central station. i would defintely come back! thanks!
1,7/9/2012,Gregory,"If you want the authentic Amsterdam houseboat experience, this is it! It is a great boat, located on a quiet canal with a view of the Rijksmuseum towers. Spotlessly clean, comfortable bed, hot shower, good wifi signal, nice kitchen with fridge, cofee maker, and stove for cooking meals. The location is very accessible to museums, gym bar and bakery on the corner, nine streets area with lots of resaraunts and bars and jazz clubs, Vondelpark, shopping on Utrechtstraat, and all of the sites of central Amsterdam. \r<br/>\r<br/>Now for the special part: If you are at all interested in boats and maritime history, this one is a great find. The cabin where guests stay is actually the cargo hold within the original hull of a freight boat that once hauled goods all over Holland. (Just imagine the countless tons of cargo that came in and out of the place where you are sleeping!) And yet the interior beautifully and comfortably remodeled using new and salvaged materials including a number of hatches that serve as skylights. The original steerhouse is still intact, along with the original one-cylinder engine down below. But the highlight is the captain's quarters where the family who owned and operated the boat lived during the 1920s (not open for guests to sleep in, yet). Stepping down into the quarters, complete with original cabinets and cookstove, feels like traveling back in time. The host cares a lot about preserving the historical character of his boat, because most other boats like it have long since gone to the scrapyard. He showed us fascinating photographs of the original owners who lived on the boat when it plied the canals nearly 100 years ago. \r<br/>\r<br/>There are many houseboats in Amsterdam. A lot of them are glorified trailer houses on flat barges. This one is a gem.",en,2,0,0.01323,306,"if you want the authentic amsterdam houseboat experience, this be it! it be a great boat, locate on a quiet canal with a view of the rijksmuseum towers. spotlessly clean, comfortable bed, hot shower, good wifi signal, nice kitchen with fridge, cofee maker, and stave for cook meals. the location be very accessible to museums, gym bar and bakery on the corner, nine street area with lot of resaraunts and bar and jazz clubs, vondelpark, shopping on utrechtstraat, and all of the site of central amsterdam. now for the special part: if you be at all interested in boat and maritime history, this one be a great find. the cabin where guest stay be actually the cargo hold within the original hull of a freight boat that once haul good all over holland. (just imagine the countless ton of cargo that come in and out of the place where you be sleeping!) and yet the interior beautifully and comfortably remodel use new and salvaged material include a number of hatch that serve a skylights. the original steerhouse be still intact, along with the original one-cylinder engine down below. but the highlight be the captain's quarter where the family who own and operate the boat live during the s (not_open_for guest to sleep in, yet). step down into the quarters, complete with original cabinet and cookstove, feel like travel back in time. the host care a lot about preserve the historical character of his boat, because most other boat like it have long since go to the scrapyard. he show u fascinate photograph of 

In [3]:
print(df.shape)

(258002, 9)


In [6]:
analyzer = SentimentIntensityAnalyzer()

In [10]:
#Testing vader with positive intent
text = 'Amazing place!!! I loved every minute of it!'
score = analyzer.polarity_scores(text)
print(score)

{'neg': 0.0, 'neu': 0.404, 'pos': 0.596, 'compound': 0.871}


In [11]:
#Testing vader with negative intent
neg_text = "Terrible place, I hated it. Would not recommend at all."
score1 = analyzer.polarity_scores(neg_text)
print(score1)

{'neg': 0.573, 'neu': 0.427, 'pos': 0.0, 'compound': -0.8559}


In [23]:
def get_sentiment(text):
    compound = analyzer.polarity_scores(str(text))['compound']

    if compound >= 0.3:
        return 'positive'
    if compound <= -0.3:
        return 'negative'
    else: 
        return 'neutral'

In [26]:
#Testing vader with this examples
print(get_sentiment("AMAZING place!!! I loved every minute of it!"))
print(get_sentiment("Terrible place, I hated it. Would not recommend at all."))
print(get_sentiment("The place was okay, nothing special."))

positive
negative
neutral


In [27]:
print(analyzer.polarity_scores("The place was okay, nothing special."))

{'neg': 0.277, 'neu': 0.49, 'pos': 0.233, 'compound': -0.092}


In [30]:
df['sentiment'] = df['comments'].progress_apply(get_sentiment)
print(df['sentiment'].value_counts())

  0%|          | 0/258002 [00:00<?, ?it/s]

sentiment
positive    249687
neutral       6801
negative      1514
Name: count, dtype: int64


## Class Distribution

The dataset is heavily skewed toward positive reviews since most guests who had a bad experience simply don't leave a review. To address this imbalance, positive reviews were downsampled to match the neutral class size.

In [31]:
positive = df[df['sentiment'] == 'positive']
neutral = df[df['sentiment'] == 'neutral']
negative = df[df['sentiment'] == 'negative']

positive_sampled = positive.sample(n = len(neutral), random_state=42)
df_balanced = pd.concat([positive_sampled, neutral, negative])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print(df_balanced['sentiment'].value_counts())

sentiment
positive    6801
neutral     6801
negative    1514
Name: count, dtype: int64


## Saving the dataset

  The balanced dataset is saved as a CSV file to be used in the next notebook for EDA.

In [32]:
df_balanced.to_csv(r"C:\Users\kumri\Downloads\Data Science Notebooks\AirBnB sentiment analysis\sentiment_tags.csv", 
                   index = False)